# Reviewer Notebook — qgrav v1.0 AISim Patches

Companion to `docs/PHYSICS_REVIEW_PACKET.md`. Reproduces every numerical claim in that document.

Estimated run time: **<2 minutes on a modern laptop**.

## Prerequisites

```bash
git clone <repo>
cd quantum-grav-platform
python -m venv .venv && source .venv/bin/activate  # (Windows: .venv\Scripts\activate)
pip install -U pip
pip install -e .
pip install matplotlib jupyter
```

Then `jupyter notebook docs/reviewer_notebook.ipynb` and run all cells.

## 1. Imports & setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from qgrav.physics.constants import WAVELENGTH_RB87_D2
from qgrav.sim_ai.aisim_adapter import (
    run_aisim_gravity_sweep,
)
from qgrav.vendor.aisim import (
    AtomicEnsemble,
    FreePropagator,
    GravityFreePropagator,
    IntensityProfile,
    SpatialSuperpositionTransitionPropagator,
    Wavevectors,
)

print("Imports OK")

## 2. Reproduce the cross-validation (§9 of the review packet)

In [ ]:
common = dict(
    n_atoms=200,
    seed=42,
    n_gravity_points=11,
    gravity_span_m_s2=4e-6,
    lock_to_midfringe=True,
)

hybrid = run_aisim_gravity_sweep(**common, gravity_propagation=False)
simulated = run_aisim_gravity_sweep(**common, gravity_propagation=True)

print("Hybrid    P3:", [f"{v:.3f}" for v in hybrid["output_port_3"]])
print("Simulated P3:", [f"{v:.3f}" for v in simulated["output_port_3"]])
print()
print("Hybrid    norm_diff:", [f"{v:+.3f}" for v in hybrid["normalized_differential_signal"]])
print("Simulated norm_diff:", [f"{v:+.3f}" for v in simulated["normalized_differential_signal"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
g = hybrid["gravity_values_m_s2"]
g_dg_microg = (g - hybrid["gravity_center_m_s2"]) * 1e6

axes[0].plot(g_dg_microg, hybrid["output_port_3"], "o-", label="Hybrid")
axes[0].plot(g_dg_microg, simulated["output_port_3"], "s-", label="Simulated")
axes[0].set_xlabel("dg (µg)")
axes[0].set_ylabel("P3")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title("Port-3 population")

axes[1].plot(g_dg_microg, hybrid["normalized_differential_signal"], "o-", label="Hybrid")
axes[1].plot(g_dg_microg, simulated["normalized_differential_signal"], "s-", label="Simulated")
axes[1].set_xlabel("dg (µg)")
axes[1].set_ylabel("Normalized diff")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_title("Normalized differential signal")
plt.tight_layout()
plt.show()

p3h, p3s = hybrid["output_port_3"], simulated["output_port_3"]
vis_h = (np.max(p3h) - np.min(p3h)) / (np.max(p3h) + np.min(p3h))
vis_s = (np.max(p3s) - np.min(p3s)) / (np.max(p3s) + np.min(p3s))
print(f"Visibility hybrid:    {vis_h:.4f}")
print(f"Visibility simulated: {vis_s:.4f}")
print(f"|ΔV|:                  {abs(vis_s - vis_h):.4f}  (test threshold 0.10)")
print(f"max |ΔP3|:             {np.max(np.abs(p3s - p3h)):.4f}  (test threshold 0.15)")

## 3. Single-atom fringe-rate measurement (§9.5)

Verifies that the simulated and hybrid fringe rates agree to ~0.2%.

In [ ]:
T = 0.26
tau = 25e-6
k1 = 2.0 * np.pi / float(WAVELENGTH_RB87_D2)
k2 = -k1
k_eff = k1 - k2
g_center = 9.81

# 1-atom ensemble at rest
psv = np.array([[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], dtype=np.float64)
atoms0 = AtomicEnsemble(psv, state_kets=[0, 0, 0, 0, 0, 1])
beam = IntensityProfile(r_profile=0.01, center_rabi_freq=2 * np.pi / (4 * tau))


def run_mz(g, chirp_rate, phase_scan, use_gravity):
    wv = Wavevectors(k1=k1, k2=k2, chirp_rate_rad_per_s2=chirp_rate)
    bs1 = SpatialSuperpositionTransitionPropagator(
        tau, n_pulse=1, n_pulses=3, intensity_profile=beam, wave_vectors=wv
    )
    mirror = SpatialSuperpositionTransitionPropagator(
        2 * tau, n_pulse=2, n_pulses=3, intensity_profile=beam, wave_vectors=wv
    )
    bs2 = SpatialSuperpositionTransitionPropagator(
        tau, n_pulse=3, n_pulses=3, intensity_profile=beam, wave_vectors=wv, phase_scan=phase_scan
    )
    prop = GravityFreePropagator(T, g_m_s2=g) if use_gravity else FreePropagator(T)
    a = bs1.propagate(atoms0)
    a = prop.propagate(a)
    a = mirror.propagate(a)
    a = prop.propagate(a)
    return float(bs2.propagate(a).state_occupation(3)[0])


def find_peak(g, chirp, use_gravity, n=73):
    phis = np.linspace(0, 2 * np.pi, n)
    p3 = np.array([run_mz(g, chirp, p, use_gravity) for p in phis])
    return phis[np.argmax(p3)]


chirp_rate = -k_eff * g_center
dgs = np.linspace(-2e-6, 2e-6, 11)
sim_peaks = np.array([find_peak(g_center + dg, chirp_rate, True) for dg in dgs])
hyb_peaks = np.array([find_peak(g_center, 0.0, False) - k_eff * dg * T**2 for dg in dgs])

sim_slope = np.polyfit(dgs, np.unwrap(sim_peaks), 1)[0]
hyb_slope = np.polyfit(dgs, np.unwrap(hyb_peaks), 1)[0]

print(f"Sim peak slope:  {sim_slope:+.3e} rad / (m/s^2)")
print(f"Hyb peak slope:  {hyb_slope:+.3e} rad / (m/s^2)")
print(f"k_eff * T^2:     {k_eff*T**2:+.3e} rad / (m/s^2)")
print(f"Ratio sim/hyb:   {sim_slope/hyb_slope:.4f}")

## 4. Verify the constant phase offset at g = g_chirp (§8.1)

Single-atom fringe at g = g_chirp should peak at phase_scan = 0 in the hybrid mode (no gravity phase) but peaks at ~2.53 rad in the simulated mode — the residual we calibrate out.

In [ ]:
phis = np.linspace(0, 2 * np.pi, 73)
p3_sim = np.array([run_mz(g_center, chirp_rate, p, True) for p in phis])
p3_hyb = np.array([run_mz(g_center, 0.0, p, False) for p in phis])

peak_sim = phis[np.argmax(p3_sim)]
peak_hyb = phis[np.argmax(p3_hyb)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(phis / np.pi, p3_hyb, "o-", label="Hybrid")
ax.plot(phis / np.pi, p3_sim, "s-", label="Simulated")
ax.axvline(
    peak_hyb / np.pi, color="C0", linestyle="--", alpha=0.5, label=f"Hyb peak: {peak_hyb:.3f}"
)
ax.axvline(
    peak_sim / np.pi, color="C1", linestyle="--", alpha=0.5, label=f"Sim peak: {peak_sim:.3f}"
)
ax.set_xlabel("phase_scan / π")
ax.set_ylabel("P3")
ax.legend(loc="lower right")
ax.set_title(f"Fringe at g = g_chirp = {g_center} m/s²  (offset = {peak_sim - peak_hyb:.3f} rad)")
ax.grid(True, alpha=0.3)
plt.show()

print(f"Sim peak: {peak_sim:.6f} rad = {peak_sim/np.pi:.6f} π")
print(f"Hyb peak: {peak_hyb:.6f} rad")
print(f"Offset:   {peak_sim - peak_hyb:.6f} rad")

## 5. Verify equation (10) vs equation (11) — the factor-of-2 prediction (§4.3)

Symbolic check that the upstream `δ·t₀` formula gives `-2·k_eff·(g-g_chirp)·T² - phase_scan` for the MZ combination, while the integrated formula gives `-k_eff·(g-g_chirp)·T² - phase_scan` (a factor of 2 smaller).

In [ ]:
# All symbolic; no AISim used.  Requires `pip install sympy` (optional).
try:
    import sympy as sp
except ImportError:
    print("sympy not installed; run `pip install sympy` and re-run this cell.")
    print()
    print("Pen-and-paper derivation (matches equations (10) and (11) of the packet):")
    print()
    print("  phi_upstream(t0) = (-v(t0)*k_eff + alpha*t0) * t0")
    print("                   = -v0*k_eff*t0 + g*k_eff*t0**2 + alpha*t0**2")
    print("  phi_patched(t0)  = -k_eff*z(t0) + 0.5*alpha*t0**2")
    print("                   = -v0*k_eff*t0 + 0.5*g*k_eff*t0**2 + 0.5*alpha*t0**2")
    print()
    print("  MZ_upstream = -phi(0) + 2*phi(T) - phi(2T) - phi_scan")
    print("              = -2*k_eff*(g - g_chirp)*T**2 - phi_scan")
    print("  MZ_patched  = -phi(0) + 2*phi(T) - phi(2T) - phi_scan")
    print("              =   -k_eff*(g - g_chirp)*T**2 - phi_scan")
    print()
    print("  Ratio:  MZ_upstream / MZ_patched = 2.   The factor-of-2 over-count.")
else:
    t, v0, g, g_chirp, k_eff_s, T_s, phi_scan = sp.symbols(
        "t v0 g g_chirp k_eff T phi_scan", positive=True, real=True
    )
    alpha_s = -k_eff_s * g_chirp
    v_t = v0 - g * t
    z_t = v0 * t - sp.Rational(1, 2) * g * t**2
    delta_t = -v_t * k_eff_s + alpha_s * t
    phi_upstream = lambda t0: delta_t.subs(t, t0) * t0
    phi_patched = lambda t0: -k_eff_s * z_t.subs(t, t0) + sp.Rational(1, 2) * alpha_s * t0**2
    t1, t2, t3 = 0, T_s, 2 * T_s
    mz_upstream = sp.simplify(
        -phi_upstream(t1) + 2 * phi_upstream(t2) - phi_upstream(t3) - phi_scan
    )
    mz_patched = sp.simplify(-phi_patched(t1) + 2 * phi_patched(t2) - phi_patched(t3) - phi_scan)
    print("Upstream MZ:  ", mz_upstream)
    print("Patched  MZ:  ", mz_patched)
    print()
    u = mz_upstream + phi_scan
    p = mz_patched + phi_scan
    print("Ratio (upstream / patched, ignoring phase_scan):", sp.simplify(u / p))

## 6. (Optional) Investigate pulse-center hypothesis (§8.3 item 1)

Replace `atoms.time` with `atoms.time + τ/2` in the chirp term of `imprint_phase` and see if the calibration offset shrinks. This is a one-line patch to `vendor/aisim/prop.py` that the reviewer can apply temporarily.

In [ ]:
# Edit src/qgrav/vendor/aisim/prop.py, find:
#     imprint_phase = -k_eff * z_at_pulse + 0.5 * chirp * atoms.time**2
# Replace with:
#     t_center = atoms.time + self.time_delta / 2.0
#     imprint_phase = -k_eff * z_at_pulse + 0.5 * chirp * t_center**2
# Save, reload the kernel, re-run cell 4 above. The offset should drop by ~half
# (from 2.53 rad to ~1.3 rad in our experiments).
print("See markdown cell above for instructions.")